In [18]:
import numpy as np
import pandas as pd

from pathlib import Path

from ml.embedding import Embedder
from ml.clustering import Clusterer
from papers_lab.io import Storage, PaperTransform
from papers_lab import PaperAnalysis
from eda import Visualizer

In [19]:
root = Path("papers_test")
store = Storage()

researches = []
for sub in ("ieee", "acm"):
    d = root / sub
    if d.exists():
        researches += store.load_dir(d, pattern="*.pkl", recursive=True)

len(researches)
tx = PaperTransform()
df = tx.transform(researches, normalize_keywords=True)

In [20]:
abstracts = df["abstract"].dropna()
# 1) gere embeddings (supondo uma lista de abstracts)
emb = Embedder(model_name="sentence-transformers/all-MiniLM-L6-v2")
X = emb.encode(abstracts)   # shape: (n_docs, d)

In [21]:
X = np.asarray(X)
if X.ndim == 1:
    # caso raro: só 1 documento -> vira (1, d)
    X = X.reshape(1, -1)

# 2) KMeans
clu = Clusterer(method="kmeans", n_clusters=3, reducer="None")  # PCA fallback automático
labels = clu.fit_predict(X)

print("Resumo:", clu.summary(labels))
print("Silhouette:", clu.silhouette(X, labels))
print("Exemplos por cluster:", clu.exemplars(X, labels, k=3))
#clu.plot_2d(X, labels, title="KMeans nos embeddings")

Resumo: {0: 1, 1: 3, 2: 5}
Silhouette: 0.026163697242736816
Exemplos por cluster: {0: [3], 1: [5, 8, 7], 2: [1, 2, 4]}


In [22]:
# 3) HDBSCAN (robusto a ruído)
clu_h = Clusterer(method="hdbscan", min_cluster_size=2, reducer="umap")
labels_h = clu_h.fit_predict(X)
print("Resumo HDBSCAN:", clu_h.summary(labels_h))
print("Silhouette HDBSCAN:", clu_h.silhouette(X, labels_h))  # pode retornar None se 1 cluster
clu_h.plot_2d(X, labels_h, title="HDBSCAN nos embeddings")

RuntimeError: hdbscan não disponível — instale `hdbscan` ou use method='kmeans'.